# ☀️ Java: ERA5-Land × LCZ × Canopy × CORDEX (Tmax_annual_p95)

Dataset stack:
1) LCZ (WUDAPT/LCZ-Global, 100 m)
2) Canopy (height/presence)
3) ERA5-Land daily Tmax (baseline)
4) CORDEX Tmax_annual_p95 (baseline & future)

Outputs: LCZ thermal modifiers (air-temp based), 100 m CORDEX Tmax95 (baseline/future/Δ), and canopy–Tmax relations by LCZ.


In [2]:
!pip install geopandas rasterio rioxarray xarray numpy pandas shapely pyproj matplotlib scipy

In [ ]:
import cdsapi

dataset = "derived-era5-land-daily-statistics"
request = {
    "variable": ["2m_temperature"],
    "year": "2002",
    "month": "10",
    "day": ["05", "07"],
    "daily_statistic": "daily_mean",
    "time_zone": "utc+00:00",
    "frequency": "1_hourly",
    "area": [-5, 104, -9, 115]
}

client = cdsapi.Client()
output_file = "era5land_1951_08_java.nc"
client.retrieve(dataset, request, output_file).download()
print(f"File saved as: {output_file}")

In [9]:
import xarray as xr

java_era5 = xr.open_dataset("20dfe9e50e466a0043e0c766b13001fa.nc")
java_era5

<xarray.Dataset> Size: 38kB
Dimensions:     (valid_time: 2, latitude: 41, longitude: 111)
Coordinates:
    number      int64 8B ...
  * latitude    (latitude) float64 328B -5.0 -5.1 -5.2 -5.3 ... -8.8 -8.9 -9.0
  * longitude   (longitude) float64 888B 104.0 104.1 104.2 ... 114.8 114.9 115.0
  * valid_time  (valid_time) datetime64[ns] 16B 2002-10-05 2002-10-07
Data variables:
    t2m         (valid_time, latitude, longitude) float32 36kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-04T10:16 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
from pathlib import Path
DATA = Path("./data"); OUT = Path("./outputs"); OUT.mkdir(parents=True, exist_ok=True)
paths = {
  "boundary_java": DATA / "Java_Coastline/Shp_Garis_Pantai.shp",
  "lcz": DATA / "LCZ_Subset_Java.tif",
  "canopy": DATA / "canopy/java_canopy_height.tif",
  "era5_tmax_daily": DATA / "era5/era5land_tmax_daily_2000_2014.nc",
  "cordex_tmax95_baseline": DATA / "cordex/tmax95_baseline.nc",
  "cordex_tmax95_future":   DATA / "cordex/tmax95_future.nc",
}
params = {"crs_work":"EPSG:23845","grid_res_m":100,"canopy_min_height_m":3,"ref_lcz_natural":[11,12,13,14]}
paths

In [10]:
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr, rioxarray as rxr, rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
from scipy.stats import linregress

def reproject_match(src_da, ref_da, resampling=Resampling.nearest):
    return src_da.rio.reproject_match(ref_da, resampling=resampling)
def make_master_grid(boundary_gdf, res_m=100, crs="EPSG:23845"):
    xmin, ymin, xmax, ymax = boundary_gdf.to_crs(crs).total_bounds
    nx, ny = int(np.ceil((xmax-xmin)/res_m)), int(np.ceil((ymax-ymin)/res_m))
    transform = rasterio.transform.from_origin(xmin, ymax, res_m, res_m)
    da = xr.DataArray(np.zeros((ny,nx), dtype=np.float32), dims=("y","x"))
    return da.rio.write_transform(transform).rio.write_crs(crs)
def k_to_c(da): return da - 273.15
def daily_to_period_stat(tmax_daily, fn="mean"):
    return tmax_daily.mean("time", skipna=True) if fn=="mean" else tmax_daily.quantile(0.95, dim="time", skipna=True)


Matplotlib is building the font cache; this may take a moment.


In [ ]:
java = gpd.read_file(paths["boundary_java"]) ; ref = make_master_grid(java, res_m=params["grid_res_m"], crs=params["crs_work"]) ; ref

In [ ]:
lcz = reproject_match(rxr.open_rasterio(paths["lcz"], masked=True).squeeze(), ref, Resampling.nearest)
canopy = reproject_match(rxr.open_rasterio(paths["canopy"], masked=True).squeeze(), ref, Resampling.bilinear)
canopy_frac = (canopy >= params["canopy_min_height_m"]).astype("float32")
print("LCZ & canopy aligned.")

In [ ]:
era = xr.open_dataset(paths["era5_tmax_daily"]) ; varname = list(era.data_vars)[0]
tmax_daily_k = era[varname]
if set(['latitude','longitude']).issubset(set(tmax_daily_k.coords)):
    minx, miny, maxx, maxy = java.to_crs("EPSG:4326").total_bounds
    tmax_daily_k = tmax_daily_k.sel(longitude=slice(minx, maxx), latitude=slice(maxy, miny))
tmax_base_mean_c = k_to_c(daily_to_period_stat(tmax_daily_k, fn="mean")).rio.write_crs("EPSG:4326")
tmax_base_mean_100 = tmax_base_mean_c.rio.reproject(ref.rio.crs, resampling=Resampling.bilinear).rio.reproject_match(ref)
print("ERA5-Land baseline to 100 m ready.")

In [ ]:
classes = [int(c) for c in np.unique(lcz.values[~np.isnan(lcz.values)])]
means = {}
for c in classes:
    sel = xr.where(lcz==c, 1.0, np.nan)
    means[c] = float(np.nanmean((tmax_base_mean_100 * sel).values))
ref_classes = [c for c in params["ref_lcz_natural"] if c in means]
ref_val = float(np.nanmedian([means[c] for c in ref_classes])) if ref_classes else float(np.nanmedian(list(means.values())))
lcz_mod = xr.zeros_like(lcz, dtype="float32")
for c, m in means.items(): lcz_mod = xr.where(lcz==c, m-ref_val, lcz_mod)
lcz_mod

In [ ]:
ds_b = xr.open_dataset(paths["cordex_tmax95_baseline"]) ; ds_f = xr.open_dataset(paths["cordex_tmax95_future"]) 
var_b, var_f = list(ds_b.data_vars)[0], list(ds_f.data_vars)[0]
tmax95_b_k, tmax95_f_k = ds_b[var_b], ds_f[var_f]
for da in [tmax95_b_k, tmax95_f_k]:
    if not da.rio.crs: da.rio.write_crs("EPSG:4326", inplace=True)
tmax95_b_100 = tmax95_b_k.rio.reproject(ref.rio.crs, resampling=Resampling.bilinear).rio.reproject_match(ref)
tmax95_f_100 = tmax95_f_k.rio.reproject(ref.rio.crs, resampling=Resampling.bilinear).rio.reproject_match(ref)
tmax95_b_dist = k_to_c(tmax95_b_100) + lcz_mod
tmax95_f_dist = k_to_c(tmax95_f_100) + lcz_mod
tmax95_delta  = tmax95_f_dist - tmax95_b_dist
tmax95_delta

In [ ]:
import pandas as pd
recs = []
for c in sorted(classes):
    sel = xr.where(lcz==c, 1.0, np.nan)
    def mean_of(da): return float(np.nanmean((da * sel).values))
    recs.append({"LCZ":int(c),"Tmax95_base_C":mean_of(tmax95_b_dist),"Tmax95_fut_C":mean_of(tmax95_f_dist),"Delta_C":mean_of(tmax95_delta),"Canopy_frac":mean_of(canopy_frac)})
df_lcz = pd.DataFrame(recs)
df_lcz.to_csv(OUT/"java_lcz_tmax95_era5_based_summary.csv", index=False)
df_lcz.head()

In [ ]:
from scipy.stats import linregress
x = canopy_frac.values.flatten(); y = tmax95_b_dist.values.flatten()
def fit_slope(x,y):
    m = ~np.isnan(x) & ~np.isnan(y)
    if m.sum()<50: return np.nan, np.nan
    r = linregress(x[m], y[m]); return r.slope, r.rvalue**2
rows = []
slope_g, r2_g = fit_slope(x,y); rows.append({"LCZ":"ALL","slope_C_per_canopyfrac":slope_g,"R2":r2_g})
for c in sorted(classes):
    m = (lcz.values.flatten()==c)
    s, r2 = fit_slope(x[m], y[m]); rows.append({"LCZ":int(c),"slope_C_per_canopyfrac":s,"R2":r2})
df_slopes = pd.DataFrame(rows); df_slopes["slope_C_per_10pct_canopy"] = df_slopes["slope_C_per_canopyfrac"]*0.10
df_slopes.to_csv(OUT/"java_canopy_tmax_slopes_era5_based.csv", index=False)
df_slopes.head()

In [ ]:
tmax95_b_dist.rio.to_raster(OUT/"java_cordex_tmax95_baseline_100m.tif", compress="lzw")
tmax95_f_dist.rio.to_raster(OUT/"java_cordex_tmax95_future_100m.tif", compress="lzw")
tmax95_delta.rio.to_raster(OUT/"java_cordex_tmax95_delta_100m.tif", compress="lzw")
lcz.rio.to_raster(OUT/"java_lcz_100m.tif", compress="lzw")
canopy_frac.rio.to_raster(OUT/"java_canopy_frac_100m.tif", compress="lzw")
print("Saved outputs to:", OUT)